## Scenario

Suppose you have a Sales Delta table with 10 million records.

**Columns**:

Order_ID
Customer_ID
Product_ID
Country
Order_Date
Amount

**Users frequently execute**:

SELECT *
FROM sales
WHERE Customer_ID = 1050;

The query is slow because Spark scans many files.

In [0]:
from pyspark.sql.functions import *

data = [
    (1,101,"Laptop","India",50000),
    (2,102,"Mobile","USA",30000),
    (3,103,"Tablet","India",25000),
    (4,101,"Mouse","India",1000),
    (5,104,"Laptop","UK",55000),
]

columns = ["order_id","customer_ID","Product","Country","Amount"]

df = spark.createDataFrame(data,columns)
df.write.format("delta").mode("overwrite").saveAsTable("workspace.optimize.sales")

In [0]:
%sql
SELECT * FROM workspace.optimize.sales WHERE Customer_ID = 101;
-- Spark reads multiple data files because the records are not stored together.

**Optimize the Table**

This compacts many small files into fewer larger files.

In [0]:
%sql
OPTIMIZE workspace.optimize.sales;

**Apply Z-Ordering**

In [0]:
%sql
OPTIMIZE WORKSPACE.OPTIMIZE.SALES
ZORDER BY (CUSTOMER_ID)

### What happens?

**Before:**

**File 1**
101
102
104

**File 2**
103
101
105

**File 3**
102
104
101

Customer 101 is scattered across multiple files.

**After Z-Ordering:**

**File 1**
101
101
101

**File 2**
102
102
103

**File 3**
104
104
105

Now, Spark reads only the file containing **Customer_ID = 101** instead of scanning every file.

In [0]:
%sql
EXPLAIN
SELECT * FROM WORKSPACE.OPTIMIZE.SALES WHERE CUSTOMER_ID = 101;

_**After Z-Ordering, the execution plan shows fewer files being scanned because Delta Lake's data skipping can eliminate files that don't contain the requested values.**_

**When to Use**

Use Z-Ordering on columns that are:

Frequently used in WHERE clauses
Frequently used in joins
High-cardinality (many unique values)

**Examples:**

Customer_ID
Order_ID
Product_ID
Employee_ID

**Project-Based Example (Banking)**

Suppose your bank has a Transactions Delta table with 500 million records.

**Columns**:

Transaction_ID
Account_ID
Branch_ID
Transaction_Date
Amount

Analysts often run:

`SELECT *
FROM transactions
WHERE Account_ID = 567890;`

**To optimize:**

OPTIMIZE transactions
ZORDER BY (Account_ID);

**Result**: Records for the same Account_ID are stored closer together, enabling Delta Lake to skip unrelated files. This reduces I/O, speeds up queries, and lowers compute costs—especially beneficial for very large datasets.